# Kompletny przewodnik: Folium (mapy interaktywne w Pythonie)

Przewodnik referencyjny (Python 3.10+, folium ≥ 0.15, GeoPandas, branca). Folium generuje **mapy Leaflet.js jako HTML**: Python przygotowuje dane i konfigurację, a przeglądarka rysuje mapę (potrzebny internet do kafelków podkładu). Notebook używa jednego zbioru danych (sieć sklepów w Polsce + prawdziwe granice województw), więc przykłady są spójne ze sobą i ze stackiem Python + GeoPandas + SQL.

Każda komórka kodu jest samodzielna po wykonaniu sekcji **0**. Kod jest „inline" (bez funkcji). W tabelach parametrów ✅ = wymagany, ⬜ = opcjonalny. Każdy element zaczyna się od **„Wersji podstawowej"** (minimum kodu), a dopiero potem są parametry i wersje rozszerzone. **Legendy i kontrolki** mają przypisane rogi mapy tak, żeby się nie nakładały (zasady w sekcji 7).

**Spis treści**

0. Setup, dane przykładowe, układ współrzędnych
1. Mapa: `folium.Map`, podkłady (tiles), kontrolki, zapis
2. Markery, popupy i tooltipy
3. Kształty: linie, poligony, okręgi, bufory
4. Warstwy i `LayerControl`
5. GeoJson i GeoDataFrame (w tym `gdf.explore()`)
6. Choropleth (mapy tematyczne)
7. Kolory i legendy (bez nakładania na mapę)
8. Klastry markerów i duże zbiory
9. Heatmapy i czas
10. Wtyczki (`folium.plugins`)
11. Tytuł, CSS/JS, raster, układ wielu map
12. Integracje: GeoPandas/CRS, QGIS, SQL Server, BigQuery, Airflow
13. Zapis, wydajność, pułapki
14. Ściągawka i porównanie z innymi bibliotekami

## 0. Setup i dane przykładowe

In [ ]:
import io
import json
import base64
import urllib.request

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import folium
import folium.plugins as plugins
import branca
import branca.colormap as cm
from branca.element import MacroElement, Template

print(folium.__version__, branca.__version__, gpd.__version__)

### Układ współrzędnych: najważniejsza zasada

| Zagadnienie | Reguła |
|---|---|
| **Kolejność współrzędnych w Folium** | **`[lat, lon]`** (szerokość, długość), np. Warszawa `[52.23, 21.01]` |
| Kolejność w GeoJSON / Shapely / GeoPandas | **`(lon, lat)`** = `(x, y)`; `Point(21.01, 52.23)` |
| CRS danych przekazywanych do Folium | **EPSG:4326** (WGS84). GeoDataFrame w innym CRS: `gdf.to_crs(4326)` |
| CRS do obliczeń w metrach (odległości, bufory, pola) | Układ rzutowany: dla Polski **EPSG:2180** (PL-1992) lub lokalnie 2176–2179 (PL-2000); wynik wróć do 4326 |

Pomyłka kolejności `lat/lon` jest najczęstszym błędem: markery lądują na Oceanie Indyjskim lub w Afryce.

### Dane przykładowe

Dane są syntetyczne (sklepy losowo rozmieszczone wokół 12 miast), a granice województw prawdziwe (pobierane z GitHuba).

In [ ]:
rng = np.random.default_rng(42)

cities = pd.DataFrame({
    "city": ["Warszawa", "Kraków", "Łódź", "Wrocław", "Poznań", "Gdańsk", "Szczecin", "Lublin", "Katowice", "Białystok", "Rzeszów", "Olsztyn"],
    "lat":  [52.2297, 50.0647, 51.7592, 51.1079, 52.4064, 54.3520, 53.4285, 51.2465, 50.2649, 53.1325, 50.0412, 53.7784],
    "lon":  [21.0122, 19.9450, 19.4560, 17.0385, 16.9252, 18.6466, 14.5528, 22.5684, 19.0238, 23.1688, 21.9991, 20.4801],
    "population_k": [1860, 800, 670, 640, 540, 470, 400, 330, 290, 295, 195, 170],     # tys., przybliżone
})

# --- sklepy (punkty)
n = 400
idx = rng.choice(len(cities), n, p=cities["population_k"] / cities["population_k"].sum())
stores = pd.DataFrame({
    "store_id": [f"S{i:04d}" for i in range(n)],
    "city": cities["city"].to_numpy()[idx],
    "lat": cities["lat"].to_numpy()[idx] + rng.normal(0, 0.045, n),
    "lon": cities["lon"].to_numpy()[idx] + rng.normal(0, 0.07, n),
    "format": rng.choice(["Mały", "Średni", "Duży"], n, p=[.5, .35, .15]),
    "opened": (pd.Timestamp("2015-01-01") + pd.to_timedelta(rng.integers(0, 365 * 10, n), unit="D")).strftime("%Y-%m-%d"),
})
fmt_mult = stores["format"].map({"Mały": 1.0, "Średni": 2.2, "Duży": 4.5}).to_numpy()
stores["revenue"] = (rng.lognormal(5.5, 0.4, n) * fmt_mult).round(1)        # tys. PLN rocznie
stores["rating"] = np.clip(rng.normal(4.0, 0.5, n), 1, 5).round(1)
stores["visits"] = rng.poisson(1200 * fmt_mult)
stores["is_open"] = rng.random(n) > 0.08

gdf_stores = gpd.GeoDataFrame(stores, geometry=gpd.points_from_xy(stores["lon"], stores["lat"]), crs="EPSG:4326")

# --- granice województw (prawdziwe dane); równoważne: gpd.read_file(url)
url = "https://raw.githubusercontent.com/ppatrzyk/polska-geojson/master/wojewodztwa/wojewodztwa-medium.geojson"
voiv = gpd.read_file(io.BytesIO(urllib.request.urlopen(url, timeout=30).read()))
voiv["geometry"] = voiv.geometry.simplify(0.005, preserve_topology=True)      # mniej wierzchołków = lżejsza mapa (zob. sekcja 13)
voiv = voiv[["nazwa", "geometry"]]

# --- agregacja punktów do poligonów (spatial join): przychód i liczba sklepów w województwie
joined = gpd.sjoin(gdf_stores, voiv, how="left", predicate="within")
agg = (joined.groupby("nazwa").agg(n_stores=("store_id", "size"), revenue_sum=("revenue", "sum"),
                                   revenue_mean=("revenue", "mean"), rating_mean=("rating", "mean")).round(1).reset_index())
voiv_stats = voiv.merge(agg, on="nazwa", how="left")          # województwa bez sklepów dostają NaN (przydatne w sekcji 6)

# --- zdarzenia w czasie (wizyty): do heatmap i animacji
n_ev = 3000
ev_idx = rng.choice(len(cities), n_ev, p=cities["population_k"] / cities["population_k"].sum())
events = pd.DataFrame({
    "lat": cities["lat"].to_numpy()[ev_idx] + rng.normal(0, 0.06, n_ev),
    "lon": cities["lon"].to_numpy()[ev_idx] + rng.normal(0, 0.09, n_ev),
    "date": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n_ev), unit="D"),
    "weight": rng.integers(1, 6, n_ev),
})
events["month"] = events["date"].dt.to_period("M").astype(str)

# --- trasy: Warszawa → pozostałe miasta
waw = cities.iloc[0]
routes = [{"to": r.city, "coords": [[waw.lat, waw.lon], [r.lat, r.lon]]} for r in cities.iloc[1:].itertuples()]

POLAND = [52.0, 19.4]          # środek mapy do widoku całego kraju

# --- kolory formatów sklepów (stałe w całym przewodniku) i szablon legendy jako kontrolki Leaflet (sekcje 7, 9)
fmt_color = {"Mały": "#2a9d8f", "Średni": "#e9c46a", "Duży": "#e76f51"}
color_by_format = {"Mały": "green", "Średni": "orange", "Duży": "red"}              # nazwy kolorów dla folium.Icon
legend_template = """
{% macro script(this, kwargs) %}
var {{ this.get_name() }} = L.control({position: '__POS__'});
{{ this.get_name() }}.onAdd = function (map) {
    var div = L.DomUtil.create('div');
    div.innerHTML = __HTML__;
    L.DomEvent.disableClickPropagation(div);          // klik w legendę nie przesuwa mapy
    return div;
};
{{ this.get_name() }}.addTo({{ this._parent.get_name() }});
{% endmacro %}
"""
box_style = "background:rgba(255,255,255,.92);padding:8px 10px;border-radius:6px;box-shadow:0 1px 6px rgba(0,0,0,.3);font:12px/1.5 Arial;color:#222"
print(len(stores), "sklepów;", voiv.shape[0], "województw;", voiv_stats["revenue_sum"].isna().sum(), "bez sklepów (NaN)")
stores.head()

> Kolumny dat zapisuj w danych dla Folium jako **tekst** (`strftime`), bo obiekty `Timestamp` i `NaN` psują serializację do JSON/GeoJSON (zob. pułapki w sekcji 13).

## 1. Mapa: `folium.Map`, podkłady, kontrolki, zapis

#### Wersja podstawowa (wymagane: nic; zwykle `location` i `zoom_start`)

In [ ]:
m = folium.Map(location=[52.23, 21.01], zoom_start=11)        # [lat, lon]; Warszawa
m

W notebooku wystarczy zostawić obiekt `m` w ostatniej linii komórki, aby go wyświetlić.

### 1.1. `folium.Map`: parametry

| Parametr | Opis |
|---|---|
| `location` ⬜ | `[lat, lon]` środka mapy (bez niego `[0, 0]`; użyj `fit_bounds`, gdy znasz zakres danych) |
| `zoom_start` ⬜ | Poziom przybliżenia: 4 = kontynent, 6 = kraj, 10 = miasto, 14 = dzielnica, 17 = budynki |
| `tiles` ⬜ | Podkład: nazwa (`"OpenStreetMap"`, `"Esri.WorldImagery"`, `"OpenTopoMap"`), URL szablonu XYZ (wymaga `attr`) lub `None` (pusty) |
| `attr` ⬜ | Tekst atrybucji (obowiązkowy dla własnych kafelków) |
| `min_zoom`, `max_zoom` ⬜ | Ograniczenia przybliżenia (0–18) |
| `max_bounds` ⬜ | `True` + `min_lat`, `max_lat`, `min_lon`, `max_lon`: blokada przesuwania poza obszar |
| `width`, `height` ⬜ | Rozmiar: liczba (px) lub tekst z `%` (`"100%"`) |
| `control_scale` ⬜ | Skala mapy (podziałka) w lewym dolnym rogu |
| `zoom_control` ⬜ | `False` ukrywa przyciski +/−; `"topleft"`… zmienia pozycję |
| `prefer_canvas` ⬜ | `True` rysuje wektory na `<canvas>` (szybsze przy tysiącach `CircleMarker`) |
| `no_touch`, `disable_3d` ⬜ | Zachowanie na urządzeniach dotykowych |
| `crs` ⬜ | `"EPSG3857"` (domyślnie), `"EPSG4326"`, `"Simple"` (płaszczyzna, np. plan hali) |
| `png_enabled` ⬜ | Eksport do PNG (wymaga Selenium) |

> **Podkłady (tiles) w 2026:** `"OpenStreetMap"` działa bez klucza. Kafelki **CartoDB** (`"CartoDB positron"`, `"dark_matter"`) wymagają teraz **klucza API** (Folium zgłasza ostrzeżenie). Bez klucza używaj: `"OpenStreetMap"`, `"OpenStreetMap.HOT"`, `"OpenTopoMap"`, `"Esri.WorldImagery"` (satelita), `"Esri.WorldGrayCanvas"` (jasnoszary, dobre tło dla danych), `"Esri.WorldTopoMap"`, `"Esri.WorldStreetMap"`. Zawsze sprawdzaj warunki użycia dostawcy przy publikacji.

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas",      # jasne tło: dane lepiej widoczne
               control_scale=True, min_zoom=5, max_zoom=16, prefer_canvas=True,
               width="100%", height=550)
m

### 1.2. Wiele podkładów i przełączanie (`TileLayer`)

| Parametr `folium.TileLayer` | Opis |
|---|---|
| `tiles` ✅ | Nazwa podkładu lub URL `https://.../{z}/{x}/{y}.png` |
| `attr` ⬜ | Atrybucja (wymagana przy własnym URL) |
| `name` ⬜ | Nazwa w `LayerControl` |
| `overlay` ⬜ | `False` = podkład (przełącznik radio), `True` = nakładka (checkbox) |
| `control` ⬜ | Czy pokazać w `LayerControl` |
| `show` ⬜ | Czy widoczna na starcie |
| `opacity`, `min_zoom`, `max_zoom`, `subdomains` ⬜ | |

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles=None)         # tiles=None: dodajemy własne podkłady poniżej
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer("Esri.WorldGrayCanvas", name="Szary").add_to(m)
folium.TileLayer("Esri.WorldImagery", name="Satelita").add_to(m)
folium.TileLayer("OpenTopoMap", name="Topograficzna").add_to(m)

# własny podkład XYZ (np. serwer kafelków firmy) wymaga attr
folium.TileLayer(tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png", attr="© OpenStreetMap contributors",
                 name="Własny XYZ", show=False).add_to(m)

# usługa WMS (np. ortofotomapa Geoportalu GUGiK); sprawdź aktualność adresu i warunki użycia
folium.WmsTileLayer(url="https://mapy.geoportal.gov.pl/wss/service/PZGIK/ORTO/WMS/StandardResolution",
                    layers="Raster", fmt="image/png", transparent=False, name="Ortofotomapa (GUGiK WMS)",
                    attr="© GUGiK", overlay=False, show=False).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

### 1.3. Dopasowanie widoku do danych, ograniczenia i zapis

In [ ]:
m = folium.Map(tiles="OpenStreetMap")                             # bez location i zoom: ustawimy fit_bounds
sw = [gdf_stores["lat"].min(), gdf_stores["lon"].min()]            # południowy zachód [lat, lon]
ne = [gdf_stores["lat"].max(), gdf_stores["lon"].max()]            # północny wschód
m.fit_bounds([sw, ne], padding=(20, 20), max_zoom=8)               # padding w px: ✅ bounds, ⬜ padding/max_zoom
# z GeoDataFrame: minx, miny, maxx, maxy = gdf.total_bounds → [[miny, minx], [maxy, maxx]]
m

In [ ]:
# zapis do pliku HTML (samodzielny plik do udostępnienia; kafelki ładują się z internetu)
m.save("mapa.html")

# fragment HTML do wstawienia w stronę/raport: m.get_root().render() lub m._repr_html_()
html = m.get_root().render()
print(len(html) // 1024, "KB")

## 2. Markery, popupy i tooltipy

### 2.1. `folium.Marker`

**Kiedy:** pojedyncze, ważne punkty (≤ kilkaset). Do tysięcy punktów użyj `CircleMarker` (canvas), klastrów lub heatmapy (sekcje 8–9).

#### Wersja podstawowa (wymagane: `location`)

In [ ]:
m = folium.Map(location=[52.23, 21.01], zoom_start=12)
folium.Marker(location=[52.2297, 21.0122]).add_to(m)               # najprostszy marker
m

# z podpowiedzią (tooltip: po najechaniu) i okienkiem (popup: po kliknięciu)
m = folium.Map(location=[52.23, 21.01], zoom_start=12)
folium.Marker([52.2297, 21.0122], tooltip="Warszawa", popup="Centrum Warszawy").add_to(m)
m

| Parametr `folium.Marker` | Opis |
|---|---|
| `location` ✅ | `[lat, lon]` |
| `popup` ⬜ | Tekst, `folium.Popup` lub obiekt HTML/IFrame |
| `tooltip` ⬜ | Tekst lub `folium.Tooltip` |
| `icon` ⬜ | `folium.Icon`, `DivIcon`, `CustomIcon`, `BeautifyIcon` |
| `draggable` ⬜ | `True` = marker można przesuwać |
| `tags` ⬜ | Lista tagów dla wtyczki `TagFilterButton` (sekcja 10) |

### 2.2. Ikony: kolor, symbol, zależność od danych

| Parametr `folium.Icon` | Opis |
|---|---|
| `color` ⬜ | Kolor markera: `red`, `blue`, `green`, `orange`, `purple`, `darkred`, `lightred`, `beige`, `darkblue`, `darkgreen`, `cadetblue`, `darkpurple`, `white`, `pink`, `lightblue`, `lightgreen`, `gray`, `black`, `lightgray` |
| `icon_color` ⬜ | Kolor symbolu wewnątrz (`"white"`, hex) |
| `icon` ⬜ | Nazwa symbolu (Font Awesome przy `prefix="fa"`, Bootstrap przy `prefix="glyphicon"`) |
| `prefix` ⬜ | `"glyphicon"` (domyślnie) lub `"fa"` |
| `angle` ⬜ | Obrót ikony w stopniach |

In [ ]:
icon_by_format = {"Mały": "shopping-basket", "Średni": "shopping-cart", "Duży": "building"}

top = gdf_stores.nlargest(25, "revenue")                          # tylko 25 największych (markery są ciężkie)
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
for r in top.itertuples():
    folium.Marker(
        location=[r.lat, r.lon],
        tooltip=f"{r.store_id} · {r.city}",
        icon=folium.Icon(color=color_by_format[r.format], icon=icon_by_format[r.format], prefix="fa", icon_color="white"),
    ).add_to(m)
m

### 2.3. `folium.CircleMarker` i `folium.Circle`

| Różnica | `CircleMarker` | `Circle` |
|---|---|---|
| Rozmiar | `radius` w **pikselach** (stały przy zoomie) | `radius` w **metrach** (rośnie z przybliżeniem) |
| Zastosowanie | Wykres bąbelkowy (rozmiar = wartość), wiele punktów | Strefa zasięgu (np. 5 km od sklepu) |

#### Wersja podstawowa (wymagane: `location`, `radius`)

In [ ]:
m = folium.Map(location=[52.23, 21.01], zoom_start=11)
folium.CircleMarker(location=[52.23, 21.01], radius=10).add_to(m)                 # 10 px
folium.Circle(location=[52.23, 21.01], radius=5000).add_to(m)                     # 5 km (metry)
m

| Styl (`**kwargs` Leaflet Path) | Opis |
|---|---|
| `color` | Kolor obrysu |
| `weight` | Grubość obrysu (px) |
| `opacity` | Przezroczystość obrysu (0–1) |
| `fill` | `True`/`False` |
| `fill_color` | Kolor wypełnienia (domyślnie jak `color`) |
| `fill_opacity` | Przezroczystość wypełnienia |
| `dash_array` | Obrys przerywany, np. `"5, 5"` |
| `popup`, `tooltip` | Jak w `Marker` |

In [ ]:
# wykres bąbelkowy: rozmiar = przychód, kolor = format
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas", prefer_canvas=True)
for r in gdf_stores.itertuples():
    folium.CircleMarker(
        location=[r.lat, r.lon],
        radius=3 + np.sqrt(r.revenue) / 2.5,                      # sqrt: pole koła ∝ wartość (uczciwsze niż skalowanie promienia liniowo)
        color="white", weight=0.8, fill=True, fill_color=fmt_color[r.format], fill_opacity=0.75,
        tooltip=f"{r.store_id} ({r.format}): {r.revenue:,.0f} tys. PLN",
    ).add_to(m)
m

### 2.4. Popup i Tooltip: treść HTML

| Parametr `folium.Popup` | Opis |
|---|---|
| `html` ✅ | Tekst, HTML, `folium.Html` lub `folium.IFrame` |
| `max_width` ⬜ | Maks. szerokość okna w px (domyślnie 300) |
| `parse_html` ⬜ | `True` przy przekazaniu `folium.Html` |
| `show` ⬜ | `True` otwiera popup na starcie |
| `sticky` ⬜ | Popup nie zamyka się po kliknięciu poza nim |
| `lazy` ⬜ | `True` ładuje treść dopiero po kliknięciu (lżejszy plik przy wielu popupach) |

| Parametr `folium.Tooltip` | Opis |
|---|---|
| `text` ✅ | Treść (może zawierać HTML) |
| `sticky` ⬜ | `True` = tooltip podąża za kursorem |
| `style` ⬜ | Tekst CSS, np. `"background-color:white; font-size:12px;"` |

In [ ]:
# a) popup z tabelą HTML (z DataFrame) i stylizowany tooltip
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
for r in gdf_stores.nlargest(40, "revenue").itertuples():
    table = (f"<b>{r.store_id}</b> ({r.city})<table style='font-size:12px'>"
             f"<tr><td>Format</td><td><b>{r.format}</b></td></tr>"
             f"<tr><td>Przychód</td><td>{r.revenue:,.0f} tys. PLN</td></tr>"
             f"<tr><td>Ocena</td><td>{r.rating:.1f} ★</td></tr>"
             f"<tr><td>Otwarty od</td><td>{r.opened}</td></tr></table>")
    folium.CircleMarker([r.lat, r.lon], radius=6, color="#264653", fill=True, fill_opacity=0.8,
                        popup=folium.Popup(table, max_width=260, lazy=True),
                        tooltip=folium.Tooltip(f"{r.store_id}", sticky=True, style="font-size:12px;")).add_to(m)
m

In [ ]:
# b) wykres matplotlib/seaborn jako obraz w popupie (base64): np. struktura formatów sklepów w mieście
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
for c in cities.itertuples():
    sub = gdf_stores[gdf_stores["city"] == c.city]["format"].value_counts().reindex(["Mały", "Średni", "Duży"]).fillna(0)
    fig, ax = plt.subplots(figsize=(3, 2))
    ax.bar(sub.index, sub.values, color=[fmt_color[k] for k in sub.index])
    ax.set_title(f"{c.city}: formaty sklepów", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=70, bbox_inches="tight"); plt.close(fig)       # plt.close: nie wyświetla wykresu w notebooku
    img = base64.b64encode(buf.getvalue()).decode()
    html = f'<img src="data:image/png;base64,{img}" width="240">'
    folium.Marker([c.lat, c.lon], popup=folium.Popup(html, max_width=280), tooltip=c.city).add_to(m)
m

> Obrazy base64 w popupach **zwiększają plik HTML** (każdy ~10–20 KB). Dla setek obiektów użyj `lazy=True` albo generuj wykresy tylko dla kluczowych punktów. Alternatywy: `folium.VegaLite(altair_chart)` w popupie lub `IFrame` z HTML-em Plotly (`fig.to_html(full_html=False, include_plotlyjs="cdn")`).

### 2.5. `DivIcon`: dowolny HTML jako znacznik (np. etykiety tekstowe, liczby)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
for c in cities.itertuples():
    folium.Marker([c.lat, c.lon], icon=folium.DivIcon(
        html=f'<div style="font:600 11px Arial;color:#264653;text-shadow:0 0 3px white, 0 0 3px white;white-space:nowrap">{c.city}</div>',
        icon_size=(100, 20), icon_anchor=(-6, 10))).add_to(m)                # anchor przesuwa etykietę względem punktu
    folium.CircleMarker([c.lat, c.lon], radius=4, color="#264653", fill=True, fill_opacity=1).add_to(m)
m

## 3. Kształty: linie, poligony, okręgi, bufory

#### Wersja podstawowa (wymagane: lista współrzędnych)

In [ ]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)
folium.PolyLine(locations=[[52.23, 21.01], [50.06, 19.94]]).add_to(m)                    # linia Warszawa–Kraków
folium.Polygon(locations=[[53, 20], [53, 22], [52, 22], [52, 20]]).add_to(m)             # wielokąt (zamyka się automatycznie)
folium.Rectangle(bounds=[[50, 16], [51, 18]]).add_to(m)                                  # prostokąt: [[lat_min, lon_min], [lat_max, lon_max]]
m

| Obiekt | Wymagane ✅ | Uwagi |
|---|---|---|
| `folium.PolyLine(locations)` | lista `[lat, lon]` (lub lista list dla wielu linii) | `smooth_factor`, `no_clip`, `tooltip`, `popup` |
| `folium.Polygon(locations)` | lista `[lat, lon]` (lub lista pierścieni: z otworami) | `fill`, `fill_color`, `fill_opacity` |
| `folium.Rectangle(bounds)` | `[[lat_min, lon_min], [lat_max, lon_max]]` | |
| `folium.Circle(location, radius)` | środek, promień w **metrach** | |

Style (`color`, `weight`, `opacity`, `dash_array`, `fill_color`, `fill_opacity`) są wspólne dla wszystkich kształtów.

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")

# a) trasy Warszawa → miasta: grubość wg liczby ludności, linia przerywana
pop = cities.set_index("city")["population_k"]
for r in routes:
    folium.PolyLine(r["coords"], color="#264653", weight=1 + pop[r["to"]] / 250, opacity=0.7, dash_array="6, 6",
                    tooltip=f"Warszawa → {r['to']}").add_to(m)

# b) animowana linia (plugin AntPath): kierunek przepływu
plugins.AntPath(locations=routes[0]["coords"], delay=800, dash_array=[10, 20], color="#e76f51", pulse_color="#fff", weight=5).add_to(m)

# c) tekst wzdłuż linii (PolyLineTextPath)
line = folium.PolyLine(routes[4]["coords"], color="#2a9d8f", weight=3).add_to(m)
plugins.PolyLineTextPath(line, "      WAW → POZ      ", repeat=False, offset=-8, attributes={"fill": "#2a9d8f", "font-weight": "bold", "font-size": "13"}).add_to(m)

# d) obszar (Polygon) i prostokąt
folium.Polygon([[53.2, 20.2], [53.2, 21.8], [52.4, 21.8], [52.4, 20.2]], color="#e9c46a", weight=2, fill=True, fill_opacity=0.25, tooltip="Strefa północ").add_to(m)
m

### 3.1. Bufory i odległości w metrach (GeoPandas + EPSG:2180)

Okrąg o promieniu w metrach można narysować przez `folium.Circle`, ale **bufory z prawdziwych geometrii** (np. 15 km wokół każdego sklepu, suma stref) liczy się w układzie rzutowanym, a do mapy wraca się do 4326.

In [ ]:
big = gdf_stores[gdf_stores["format"] == "Duży"]
buf = big.to_crs(2180).buffer(15_000).to_crs(4326)                         # 15 km w EPSG:2180 → z powrotem WGS84
buf_union = gpd.GeoSeries([buf.union_all()], crs=4326)                     # połączone strefy (bez nakładania)

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(buf_union, name="Strefa 15 km od dużych sklepów",
               style_function=lambda f: {"fillColor": "#e76f51", "color": "#e76f51", "weight": 1, "fillOpacity": 0.25}).add_to(m)
for r in big.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=3, color="#e76f51", fill=True, fill_opacity=1).add_to(m)

# odległość każdego sklepu do najbliższego dużego sklepu (metry, EPSG:2180)
nearest = gpd.sjoin_nearest(gdf_stores.to_crs(2180), big[["store_id", "geometry"]].to_crs(2180), how="left", distance_col="dist_m", lsuffix="", rsuffix="big")
print(nearest["dist_m"].describe().round(0))
m

## 4. Warstwy i `LayerControl`

**Kiedy:** gdy na mapie jest kilka zbiorów danych (punkty, poligony, trasy) i użytkownik ma je włączać/wyłączać. `FeatureGroup` grupuje obiekty w jedną warstwę, a `LayerControl` pokazuje przełączniki.

#### Wersja podstawowa (wymagane: nic)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6)

fg = folium.FeatureGroup(name="Sklepy").add_to(m)                   # warstwa
for r in gdf_stores.head(60).itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=4).add_to(fg)         # obiekty dodajemy do warstwy, nie do mapy

folium.LayerControl().add_to(m)                                      # przełącznik warstw (dodaj NA KOŃCU)
m

| Parametr `folium.FeatureGroup` | Opis |
|---|---|
| `name` ⬜ | Nazwa w `LayerControl` |
| `overlay` ⬜ | `True` (domyślnie) = checkbox; `False` = radio (jedna aktywna) |
| `control` ⬜ | `False` = warstwa bez wpisu w `LayerControl` |
| `show` ⬜ | `False` = domyślnie wyłączona |

| Parametr `folium.LayerControl` | Opis |
|---|---|
| `position` ⬜ | `"topright"` (domyślnie), `"topleft"`, `"bottomleft"`, `"bottomright"` |
| `collapsed` ⬜ | `True` = zwinięty do ikony (oszczędza miejsce, nie zasłania mapy) |
| `autoZIndex` ⬜ | Automatyczna kolejność warstw |
| `draggable` ⬜ | Można przesuwać panel |

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer("Esri.WorldGrayCanvas", name="Szary", show=False).add_to(m)

# kolejność dodawania = kolejność rysowania: poligony najpierw (pod spodem), punkty na wierzchu
folium.GeoJson(voiv, name="Województwa", style_function=lambda f: {"fillColor": "#cfd8dc", "color": "#78909c", "weight": 1, "fillOpacity": 0.25}).add_to(m)

layers = {}
for fmt, col in fmt_color.items():                                   # jedna warstwa na format
    layers[fmt] = folium.FeatureGroup(name=f"Sklepy: {fmt}", show=(fmt != "Mały")).add_to(m)
    for r in gdf_stores[gdf_stores["format"] == fmt].itertuples():
        folium.CircleMarker([r.lat, r.lon], radius=4, color="white", weight=0.7, fill=True, fill_color=col, fill_opacity=0.85,
                            tooltip=f"{r.store_id} ({fmt})").add_to(layers[fmt])

routes_fg = folium.FeatureGroup(name="Trasy z Warszawy", show=False).add_to(m)
for r in routes:
    folium.PolyLine(r["coords"], color="#264653", weight=2).add_to(routes_fg)

folium.LayerControl(position="topright", collapsed=False).add_to(m)
m

### 4.1. Grupowanie przełączników: `GroupedLayerControl`

Gdy warstw jest dużo, pogrupuj je (np. „Punkty", „Podkłady"). Można ustawić grupę wyłącznego wyboru (`exclusive_groups=True` = jak radio).

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
fg_small = folium.FeatureGroup(name="Mały").add_to(m)
fg_mid = folium.FeatureGroup(name="Średni").add_to(m)
fg_big = folium.FeatureGroup(name="Duży").add_to(m)
for r in gdf_stores.itertuples():
    target = {"Mały": fg_small, "Średni": fg_mid, "Duży": fg_big}[r.format]
    folium.CircleMarker([r.lat, r.lon], radius=3, color=fmt_color[r.format], fill=True, fill_opacity=0.8).add_to(target)

folium.LayerControl(collapsed=True).add_to(m)                          # zwykły LayerControl (np. dla podkładów)
plugins.GroupedLayerControl(groups={"Format sklepu": [fg_small, fg_mid, fg_big]}, exclusive_groups=False, collapsed=False).add_to(m)
m

### 4.2. Podgrupy w klastrze: `FeatureGroupSubGroup`

Jeden klaster liczy obiekty z kilku warstw, a każdą warstwę można niezależnie włączać (szczegóły klastrów w sekcji 8).

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
parent = plugins.MarkerCluster(name="Wszystkie sklepy").add_to(m)
for fmt, col in fmt_color.items():
    sub = plugins.FeatureGroupSubGroup(parent, name=f"Format: {fmt}").add_to(m)
    for r in gdf_stores[gdf_stores["format"] == fmt].itertuples():
        folium.CircleMarker([r.lat, r.lon], radius=5, color=col, fill=True, fill_opacity=0.9, tooltip=r.store_id).add_to(sub)
folium.LayerControl(collapsed=False).add_to(m)
m

## 5. GeoJson i GeoDataFrame

**Kiedy:** wszystko, co ma geometrię z danych (poligony, linie, punkty z atrybutami). `folium.GeoJson` przyjmuje słownik GeoJSON, ścieżkę/URL **lub GeoDataFrame** (w EPSG:4326). Dzięki stylom zależnym od atrybutów i tooltipom to najczęściej używany obiekt w analizie przestrzennej.

#### Wersja podstawowa (wymagane: `data`)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6)
folium.GeoJson(voiv).add_to(m)                                       # GeoDataFrame wprost
m

| Parametr `folium.GeoJson` | Opis |
|---|---|
| `data` ✅ | GeoDataFrame, dict GeoJSON, string JSON, ścieżka lub URL |
| `style_function` ⬜ | `lambda feature: {...}` zwraca styl (`fillColor`, `color`, `weight`, `fillOpacity`, `dashArray`) wg atrybutów `feature["properties"]` |
| `highlight_function` ⬜ | Styl po najechaniu myszką |
| `tooltip` ⬜ | `folium.GeoJsonTooltip(fields=[...], aliases=[...])` |
| `popup` ⬜ | `folium.GeoJsonPopup(fields=[...], aliases=[...])` |
| `marker` ⬜ | Dla punktów: `folium.CircleMarker(radius=...)` lub `folium.Marker(icon=...)` zamiast domyślnej pinezki |
| `name`, `overlay`, `control`, `show` ⬜ | Parametry warstwy (jak `FeatureGroup`) |
| `zoom_on_click` ⬜ | Kliknięcie przybliża do obiektu |
| `smooth_factor` ⬜ | Uproszczenie linii przy rysowaniu (większe = szybciej, mniej szczegółów) |
| `embed` ⬜ | `False` = dane ładowane z pliku (rzadko używane) |
| `popup_keep_highlighted` ⬜ | Zostaw podświetlenie po otwarciu popupu |

| Parametr `folium.GeoJsonTooltip` / `GeoJsonPopup` | Opis |
|---|---|
| `fields` ✅ | Lista kolumn do pokazania |
| `aliases` ⬜ | Etykiety zamiast nazw kolumn |
| `localize` ⬜ | `True` formatuje liczby/daty wg ustawień przeglądarki (Tooltip) |
| `labels` ⬜ | `False` ukrywa etykiety pól |
| `sticky` ⬜ | (Tooltip) podąża za kursorem |
| `style` ⬜ | CSS (Tooltip), `max_width` (Popup) |

In [ ]:
# poligony: styl zależny od atrybutu + podświetlenie + tooltip i popup z danymi
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(
    voiv_stats, name="Województwa",
    style_function=lambda f: {"fillColor": "#2a9d8f" if f["properties"]["n_stores"] else "#cfd8dc",          # None (NaN) → szary
                              "color": "white", "weight": 1.5, "fillOpacity": 0.6},
    highlight_function=lambda f: {"weight": 3, "color": "#264653", "fillOpacity": 0.85},
    tooltip=folium.GeoJsonTooltip(fields=["nazwa", "n_stores", "revenue_sum"],
                                  aliases=["Województwo:", "Sklepy:", "Przychód [tys. PLN]:"], localize=True, sticky=True,
                                  style="font-size:12px;"),
    popup=folium.GeoJsonPopup(fields=["nazwa", "revenue_mean", "rating_mean"], aliases=["Województwo", "Śr. przychód", "Śr. ocena"]),
    zoom_on_click=True,
).add_to(m)
m

In [ ]:
# punkty z GeoDataFrame: marker = CircleMarker, kolor wg kategorii
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(
    gdf_stores[["store_id", "city", "format", "revenue", "rating", "geometry"]],           # tylko potrzebne kolumny (lżejszy plik)
    name="Sklepy",
    marker=folium.CircleMarker(radius=4, weight=0.8, color="white"),
    style_function=lambda f: {"fillColor": fmt_color[f["properties"]["format"]], "fillOpacity": 0.85, "color": "white", "weight": 0.8},
    tooltip=folium.GeoJsonTooltip(fields=["store_id", "city", "format", "revenue"], aliases=["ID", "Miasto", "Format", "Przychód"]),
).add_to(m)
m

> `style_function` dostaje obiekt GeoJSON (**po serializacji**): brakujące wartości (`NaN`) są tam `None`, a nie `NaN`. Dlatego warunek `if v is not None`.

### 5.1. Skrót z GeoPandas: `gdf.explore()`

`GeoDataFrame.explore()` (GeoPandas ≥ 0.13) tworzy mapę Folium jednym wywołaniem, z klasyfikacją, legendą i tooltipem. Wymaga `folium`, `matplotlib`, `mapclassify`. Zwraca obiekt `folium.Map`, więc można go dalej rozbudowywać.

#### Wersja podstawowa

In [ ]:
voiv_stats.explore()

In [ ]:
voiv_stats.explore(column="revenue_sum")                                             # kolor wg kolumny (legenda automatycznie)

| Parametr `gdf.explore` | Opis |
|---|---|
| `column` ⬜ | Kolumna do kolorowania (liczba = skala ciągła/klasy, tekst = kategorie) |
| `cmap` ⬜ | Paleta matplotlib (`"YlOrRd"`, `"viridis"`, `"Set2"`) lub lista kolorów |
| `scheme` ⬜ | Klasyfikacja `mapclassify`: `"quantiles"`, `"naturalbreaks"`, `"equalinterval"`, `"fisherjenks"`, `"userdefined"` (wymaga `k=` lub `classification_kwds`) |
| `k` ⬜ | Liczba klas |
| `legend` ⬜ | `True` (domyślnie) / `False` |
| `legend_kwds` ⬜ | `{"caption": "...", "colorbar": False, "scale": False}` |
| `tooltip`, `popup` ⬜ | `True`, `False` lub lista kolumn |
| `tiles` ⬜ | Podkład (jak w `folium.Map`) |
| `style_kwds` ⬜ | `{"color": "white", "weight": 1, "fillOpacity": 0.8}` |
| `marker_type`, `marker_kwds` ⬜ | Punkty: `"circle_marker"`/`"circle"`/`"marker"`, `{"radius": 5}` |
| `missing_kwds` ⬜ | Wygląd obiektów z brakiem wartości: `{"color": "lightgray", "label": "brak sklepów"}` |
| `m` ⬜ | Istniejąca mapa, na którą nakładamy warstwę (łączenie wielu `explore`) |
| `name`, `highlight`, `vmin`, `vmax`, `categorical`, `categories`, `width`, `height` ⬜ | |

In [ ]:
m = voiv_stats.explore(
    column="revenue_sum", cmap="YlOrRd", scheme="quantiles", k=4,
    tooltip=["nazwa", "n_stores", "revenue_sum"], popup=False, tiles="Esri.WorldGrayCanvas",
    style_kwds={"color": "white", "weight": 1, "fillOpacity": 0.85},
    missing_kwds={"color": "#e0e0e0", "label": "brak sklepów"},
    legend_kwds={"caption": "Przychód [tys. PLN]"}, name="Przychód wg województw")

gdf_stores.explore(m=m, column="format", cmap=["#2a9d8f", "#e9c46a", "#e76f51"], categories=["Mały", "Średni", "Duży"],    # druga warstwa na tej samej mapie
                   marker_kwds={"radius": 4}, style_kwds={"color": "white", "weight": 0.6, "fillOpacity": 0.9},
                   tooltip=["store_id", "city", "format"], legend=False, name="Sklepy")
folium.LayerControl(collapsed=True).add_to(m)
m

## 6. Choropleth (mapy tematyczne)

**Kiedy:** wartość/odsetek zagregowany do **jednostek powierzchniowych** (województwa, powiaty, gminy, siatka). **Interpretacja:** kolor = wartość w danym obszarze. **Pułapki:** (1) mapa zdominowana przez duże powierzchnie, więc dla wskaźników używaj **wartości względnych** (na mieszkańca, na km²), a nie sum bezwzględnych; (2) liczba i granice klas zmieniają obraz, więc zawsze podawaj metodę klasyfikacji; (3) brak danych ≠ zero, więc oznacz go osobnym kolorem.

### 6.1. `folium.Choropleth`

#### Wersja podstawowa (wymagane: `geo_data`, `data`, `columns`, `key_on`)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6)
folium.Choropleth(
    geo_data=voiv_stats,                                   # poligony
    data=voiv_stats,                                       # tabela z wartościami (może być osobnym DataFrame)
    columns=["nazwa", "revenue_sum"],                      # [kolumna klucza, kolumna wartości]
    key_on="feature.properties.nazwa",                     # gdzie w GeoJSON jest klucz łączący z data
    fill_color="YlOrRd",
).add_to(m)
m                                                          # województwa bez danych są domyślnie CZARNE (zob. nan_fill_color niżej)

| Parametr `folium.Choropleth` | Opis |
|---|---|
| `geo_data` ✅ | Poligony (GeoDataFrame, dict, ścieżka) |
| `data` ✅ | DataFrame/Series z wartościami (bez niego rysuje same granice) |
| `columns` ✅ | `[klucz, wartość]` |
| `key_on` ✅ | Ścieżka klucza w GeoJSON: `"feature.properties.nazwa"` lub `"feature.id"` (przy GeoDataFrame `id` = indeks) |
| `fill_color` ⬜ | Paleta ColorBrewer: `"YlOrRd"`, `"Blues"`, `"BuPu"`, `"YlGnBu"`, `"RdYlGn"`, `"PuOr"`… |
| `bins` ⬜ | Liczba klas (domyślnie 6) **lub lista progów** `[0, 100, 500, 1000, 3000]` |
| `nan_fill_color` ⬜ | Kolor obszarów bez danych (domyślnie `"black"`; ustaw jasnoszary!) |
| `nan_fill_opacity` ⬜ | Przezroczystość tych obszarów |
| `fill_opacity` ⬜ | Przezroczystość wypełnienia (0.6 domyślnie) |
| `line_color`, `line_weight`, `line_opacity` ⬜ | Obrys |
| `legend_name` ⬜ | Opis legendy (podpis) |
| `highlight` ⬜ | `True` = podświetlenie po najechaniu |
| `name`, `overlay`, `control`, `show` ⬜ | Parametry warstwy |
| `smooth_factor` ⬜ | Uproszczenie linii |
| `use_jenks` ⬜ | Klasy naturalne (wymaga pakietu `jenkspy`; zalecane `mapclassify`, zob. 6.3) |

In [ ]:
# wersja rozszerzona: własne progi, szary dla braków, highlight, tooltip z danymi
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
choro = folium.Choropleth(
    geo_data=voiv_stats, data=voiv_stats, columns=["nazwa", "revenue_sum"], key_on="feature.properties.nazwa",
    fill_color="YlOrRd", bins=[0, 3000, 6000, 12000, 25000, 60000],
    nan_fill_color="#e0e0e0", nan_fill_opacity=0.7, fill_opacity=0.8, line_color="white", line_weight=1, line_opacity=1,
    legend_name="Przychód sklepów [tys. PLN]", highlight=True, name="Przychód wg województw")
choro.add_to(m)

# Choropleth nie ma własnego tooltipu: dodajemy go do jego warstwy GeoJson
choro.geojson.add_child(folium.GeoJsonTooltip(fields=["nazwa", "n_stores", "revenue_sum"], aliases=["Województwo:", "Sklepy:", "Przychód:"], localize=True))
folium.LayerControl(collapsed=True).add_to(m)
m

### 6.2. Elastyczniej: `GeoJson` + kolormapa `branca`

Dla pełnej kontroli (własne kolory, tooltip, braki danych, wspólne style z resztą mapy) łączymy `GeoJson` z kolormapą. To wzorzec zalecany w większych projektach.

In [ ]:
vmin, vmax = voiv_stats["revenue_sum"].min(), voiv_stats["revenue_sum"].max()
cmap = cm.linear.YlOrRd_09.scale(vmin, vmax)                       # wbudowane palety: cm.linear.<Nazwa>
cmap.caption = "Przychód sklepów [tys. PLN]"

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(
    voiv_stats, name="Przychód",
    style_function=lambda f: {
        "fillColor": cmap(f["properties"]["revenue_sum"]) if f["properties"]["revenue_sum"] is not None else "#e0e0e0",   # braki: szary
        "color": "white", "weight": 1, "fillOpacity": 0.85},
    highlight_function=lambda f: {"weight": 3, "color": "#264653"},
    tooltip=folium.GeoJsonTooltip(fields=["nazwa", "n_stores", "revenue_sum", "rating_mean"],
                                  aliases=["Województwo", "Sklepy", "Przychód", "Śr. ocena"], localize=True, sticky=True),
).add_to(m)
cmap.add_to(m)                                                     # legenda w prawym górnym rogu (kontrolka Leaflet)
m

### 6.3. Klasyfikacja (klasy) przez `mapclassify` + `StepColormap`

Ciągła skala dobrze wygląda, ale **klasy** są czytelniejsze i porównywalne. `mapclassify` (ten sam co w `gdf.explore`) liczy progi: `Quantiles`, `NaturalBreaks` (Jenks), `EqualInterval`, `FisherJenks`, `StdMean`.

| Metoda | Kiedy |
|---|---|
| `EqualInterval` | Dane równomierne; łatwa do wyjaśnienia (stała szerokość klasy) |
| `Quantiles` | Rozkład skośny; w każdej klasie tyle samo obiektów (uwaga: sztucznie rozciąga różnice) |
| `NaturalBreaks` / `FisherJenks` | Szukanie naturalnych grup w danych |
| `UserDefined` | Progi merytoryczne (np. normy, cele biznesowe) |

In [ ]:
import mapclassify
import seaborn as sns

vals = voiv_stats["revenue_sum"].dropna()
clf = mapclassify.NaturalBreaks(vals, k=4)                          # ✅ y, ⬜ k (domyślnie 5)
print(clf)                                                          # przedziały i liczność klas

breaks = [vals.min()] + clf.bins.tolist()                           # k+1 progów: [min, b1, b2, b3, max]
colors = sns.color_palette("YlOrRd", 4).as_hex()                    # k kolorów
step = cm.StepColormap(colors=colors, index=breaks, vmin=breaks[0], vmax=breaks[-1], caption="Przychód [tys. PLN] (naturalne przerwy)")

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(
    voiv_stats, name="Klasy przychodu",
    style_function=lambda f: {"fillColor": step(f["properties"]["revenue_sum"]) if f["properties"]["revenue_sum"] is not None else "#e0e0e0",
                              "color": "white", "weight": 1, "fillOpacity": 0.9},
    tooltip=folium.GeoJsonTooltip(fields=["nazwa", "revenue_sum"], aliases=["Województwo", "Przychód"], localize=True),
).add_to(m)
step.add_to(m)
m

### 6.4. Wskaźnik względny (na sklep) i nakładanie punktów

In [ ]:
voiv_stats["rev_per_store"] = (voiv_stats["revenue_sum"] / voiv_stats["n_stores"]).round(1)       # wskaźnik zamiast sumy
v = voiv_stats["rev_per_store"].dropna()
cmap2 = cm.linear.viridis.scale(v.min(), v.max())
cmap2.caption = "Średni przychód na sklep [tys. PLN]"

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(voiv_stats, name="Przychód na sklep",
               style_function=lambda f: {"fillColor": cmap2(f["properties"]["rev_per_store"]) if f["properties"]["rev_per_store"] is not None else "#e0e0e0",
                                         "color": "white", "weight": 1, "fillOpacity": 0.75},
               tooltip=folium.GeoJsonTooltip(fields=["nazwa", "rev_per_store", "n_stores"], aliases=["Województwo", "Przychód / sklep", "Sklepy"])).add_to(m)
for r in gdf_stores.itertuples():                                   # punkty na wierzchu, mała przezroczystość
    folium.CircleMarker([r.lat, r.lon], radius=2.5, color="black", weight=0.3, fill=True, fill_color="white", fill_opacity=0.7).add_to(m)
cmap2.add_to(m)
m

## 7. Kolory i legendy (bez nakładania na mapę)

### 7.1. Kolormapy `branca`

| Obiekt | Opis |
|---|---|
| `cm.LinearColormap(colors, vmin, vmax, caption)` | Gradient: `colors` = lista kolorów (nazwy/hex) ✅ |
| `cm.StepColormap(colors, index, vmin, vmax)` | Klasy: `index` ma o 1 element więcej niż `colors` |
| `cm.linear.<Nazwa>` | Gotowe palety (`YlOrRd_09`, `Blues_09`, `RdYlBu_11`, `viridis`, `plasma`, `Set1_03`…); lista: `cm.linear._schemes` |
| `.scale(vmin, vmax)` | Dopasowanie zakresu do danych |
| `.to_step(n, method="quantiles"/"linear", data=...)` | Zamiana gradientu na klasy |
| `cmap(value)` | Zwraca kolor (hex) dla wartości (użycie w `style_function`) |
| `.caption`, `.add_to(m)` | Podpis i dodanie legendy do mapy |

In [ ]:
# własny gradient i konwersja na klasy
cmap = cm.LinearColormap(["#ffffcc", "#41b6c4", "#0c2c84"], vmin=1, vmax=5, caption="Ocena sklepu")
print(cmap(4.2))                                       # kolor dla wartości
cmap_step = cmap.to_step(4)                            # 4 równe klasy
cmap_step

In [ ]:
# paleta z matplotlib/seaborn → lista hex do StepColormap/LinearColormap
import matplotlib as mpl
hex_colors = [mpl.colors.to_hex(c) for c in mpl.colormaps["viridis"](np.linspace(0, 1, 5))]
print(hex_colors, sns.color_palette("rocket", 5).as_hex())

### 7.2. Zasady rozmieszczenia: żeby legenda nie nakładała się na mapę

Leaflet ma **cztery rogi** mapy. Kontrolki umieszczone w tym samym rogu **układają się jedna pod drugą** i nie nachodzą na siebie. Nakładanie powstaje, gdy legenda jest zwykłym `<div>` z `position: fixed/absolute` (to częsty przykład z internetu), bo nie „wie" o innych kontrolkach.

| Róg | Typowa zawartość (zalecany podział) |
|---|---|
| `topleft` | Zoom, `Fullscreen`, `MeasureControl`, `Search`, `Geocoder`, `Draw` |
| `topright` | `LayerControl` (zwinięty), kolormapa `branca` (gradient/klasy) |
| `bottomleft` | **Własne legendy** (kategorie, rozmiary), podziałka (`control_scale`), `MousePosition`, odtwarzacz `HeatMapWithTime` |
| `bottomright` | Atrybucja podkładu, `MiniMap` |

Reguły praktyczne:
1. **Legendy jako kontrolki Leaflet** (`MacroElement` poniżej), a nie sztywny `position: fixed`.
2. **Zwijaj `LayerControl`** (`collapsed=True`), gdy warstw jest dużo.
3. Szerokość legendy ≤ ok. 200 px; przy wielu pozycjach użyj dwóch kolumn lub pomiń rzadkie kategorie.
4. Tytuł mapy umieszczaj na środku u góry (sekcja 11), a nie w rogach z kontrolkami.
5. Przy małych mapach (np. w raporcie) nie dodawaj jednocześnie kolormapy, `MiniMap` i legendy.

### 7.3. Własna legenda jako kontrolka Leaflet (kategorie, rozmiary)

Szablon `legend_template` (zdefiniowany w sekcji 0) wstawia HTML legendy w wybranym rogu (`__POS__`). Używasz go dla dowolnej legendy: wystarczy zmienić zawartość (`__HTML__`) i `position`.

In [ ]:

# legenda kategorii (kolor = format sklepu)
rows = "".join(f'<div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;background:{c};margin-right:6px;vertical-align:middle"></span>{k}</div>'
               for k, c in fmt_color.items())
legend_html = f'<div style="{box_style}"><b>Format sklepu</b>{rows}</div>'

# legenda rozmiarów (promień ∝ sqrt(przychód))
sizes = [50, 200, 800]
size_rows = "".join(f'<div style="display:flex;align-items:center;gap:6px"><span style="display:inline-block;width:{2 * (3 + np.sqrt(s) / 2.5):.0f}px;height:{2 * (3 + np.sqrt(s) / 2.5):.0f}px;border-radius:50%;border:1.5px solid #555"></span>{s} tys. PLN</div>' for s in sizes)
size_html = f'<div style="{box_style}"><b>Przychód</b>{size_rows}</div>'

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas", prefer_canvas=True)
for r in gdf_stores.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=3 + np.sqrt(r.revenue) / 2.5, color="white", weight=0.8, fill=True,
                        fill_color=fmt_color[r.format], fill_opacity=0.75, tooltip=f"{r.store_id}: {r.revenue:,.0f}").add_to(m)

for html, pos in [(legend_html, "bottomleft"), (size_html, "bottomleft")]:          # dwie legendy w jednym rogu: układają się jedna nad drugą
    legend = MacroElement()
    legend._template = Template(legend_template.replace("__POS__", pos).replace("__HTML__", json.dumps(html)))
    legend.add_to(m)

folium.LayerControl(collapsed=True).add_to(m)
m

### 7.4. Legenda gradientowa: `branca` w prawym górnym rogu + własna w lewym dolnym

In [ ]:
v = gdf_stores["rating"]
cmap_r = cm.LinearColormap(["#d73027", "#fee08b", "#1a9850"], vmin=v.min(), vmax=v.max(), caption="Ocena sklepu (1–5)")     # kolormapa (topright)

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas", prefer_canvas=True)
for r in gdf_stores.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=4, color="white", weight=0.6, fill=True, fill_color=cmap_r(r.rating), fill_opacity=0.9,
                        tooltip=f"{r.store_id}: {r.rating:.1f} ★").add_to(m)
cmap_r.add_to(m)                                                                    # topright
folium.LayerControl(collapsed=True).add_to(m)                                       # topright, ale zwinięty i pod kolormapą (stos)
m

## 8. Klastry markerów i duże zbiory

**Kiedy:** setki–dziesiątki tysięcy punktów, które jako osobne markery zasłaniają mapę i spowalniają przeglądarkę. Klaster pokazuje liczbę punktów w okolicy, a po przybliżeniu rozbija się na pojedyncze markery. **Interpretacja:** liczba w kółku = ile obiektów jest w obszarze; to **liczności**, a nie wartość biznesowa (do sum/średnich użyj choropleth lub heatmapy ważonej).

#### Wersja podstawowa (wymagane: nic)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6)
plugins.MarkerCluster(locations=gdf_stores[["lat", "lon"]].values.tolist()).add_to(m)          # lista [lat, lon]
m

In [ ]:
# z tooltipem i popupem: dodajemy markery do klastra
m = folium.Map(location=POLAND, zoom_start=6)
mc = plugins.MarkerCluster(name="Sklepy").add_to(m)
for r in gdf_stores.itertuples():
    folium.Marker([r.lat, r.lon], tooltip=r.store_id, popup=f"{r.city}, {r.format}, {r.revenue:,.0f} tys. PLN").add_to(mc)
m

| Parametr `plugins.MarkerCluster` | Opis |
|---|---|
| `locations` ⬜ | Lista `[lat, lon]` (szybka droga bez pętli) |
| `popups`, `icons` ⬜ | Listy popupów/ikon równoległe do `locations` |
| `name`, `overlay`, `control`, `show` ⬜ | Parametry warstwy |
| `icon_create_function` ⬜ | Własny JavaScript tworzący ikonę klastra (kolor wg liczności) |
| `options` ⬜ | Słownik opcji Leaflet.markercluster, np. `{"maxClusterRadius": 60, "disableClusteringAtZoom": 12, "spiderfyOnMaxZoom": True, "showCoverageOnHover": False, "zoomToBoundsOnClick": True, "chunkedLoading": True}` |

In [ ]:
# własna ikona klastra: kolor zależny od liczby punktów; opcje: mniejszy promień klastrowania, bez obrysu pokrycia
icon_create_function = """
function(cluster) {
    var n = cluster.getChildCount();
    var c = n < 10 ? '#2a9d8f' : (n < 40 ? '#e9c46a' : '#e76f51');
    return L.divIcon({
        html: '<div style="background:' + c + ';color:#fff;border-radius:50%;width:36px;height:36px;line-height:36px;text-align:center;font:bold 12px Arial;border:2px solid #fff;box-shadow:0 0 4px rgba(0,0,0,.4)">' + n + '</div>',
        className: 'marker-cluster', iconSize: new L.Point(36, 36)});
}"""

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
mc = plugins.MarkerCluster(name="Sklepy", icon_create_function=icon_create_function,
                           options={"maxClusterRadius": 50, "showCoverageOnHover": False, "disableClusteringAtZoom": 13}).add_to(m)
for r in gdf_stores.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=5, color="white", weight=1, fill=True, fill_color=fmt_color[r.format], fill_opacity=0.9,
                        tooltip=f"{r.store_id} ({r.format})").add_to(mc)
m

### 8.1. `FastMarkerCluster`: dziesiątki tysięcy punktów

`MarkerCluster` tworzy obiekt Pythona dla każdego markera (wolne, duży HTML). `FastMarkerCluster` przekazuje do przeglądarki **samą tablicę współrzędnych** i tworzy markery w JavaScript przez funkcję `callback`. Kosztem jest brak złożonych popupów (tylko to, co zbudujesz w JS).

| Parametr `plugins.FastMarkerCluster` | Opis |
|---|---|
| `data` ✅ | Lista list: `[lat, lon]` lub `[lat, lon, ...dodatkowe pola]` |
| `callback` ⬜ | JS `function (row) {...; return marker;}`: `row[0]`=lat, `row[1]`=lon, kolejne pola do tooltipu |
| `name`, `overlay`, `control`, `show`, `options` ⬜ | Jak w `MarkerCluster` |

In [ ]:
callback = """
function (row) {
    var marker = L.circleMarker(new L.LatLng(row[0], row[1]), {radius: 4, color: 'white', weight: 1, fillColor: '#264653', fillOpacity: 0.85});
    marker.bindTooltip('Waga: ' + row[2]);
    return marker;
};"""

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.FastMarkerCluster(data=events[["lat", "lon", "weight"]].values.tolist(), callback=callback, name=f"Zdarzenia ({len(events):,})").add_to(m)
folium.LayerControl(collapsed=True).add_to(m)
m

> Reguła kciuka: do ~500 obiektów zwykłe markery, do ~5 000 `MarkerCluster`, powyżej `FastMarkerCluster` / `CircleMarker` na canvasie (`prefer_canvas=True`) / heatmapa; od ~100 000 punktów rozważ agregację do siatki (H3, kwadraty) i choropleth lub inną bibliotekę (pydeck/kepler, zob. sekcja 14).

## 9. Heatmapy i czas

### 9.1. `plugins.HeatMap`

**Kiedy:** gęstość punktów (gdzie jest „dużo") lub ważona wartość (gdzie jest „dużo wartości"), gdy liczy się wzorzec, a nie pojedyncze obiekty. **Interpretacja:** intensywność koloru = lokalna gęstość/suma wag; skala jest **względna** (zależy od zoomu, `radius` i `blur`), więc nie odczytujesz liczb. **Pułapka:** heatmapa z surowych punktów odzwierciedla po prostu rozmieszczenie ludności; porównuj wskaźniki względne (sklepy na mieszkańca).

#### Wersja podstawowa (wymagane: `data`)

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.HeatMap(data=events[["lat", "lon"]].values.tolist()).add_to(m)               # lista [lat, lon]
m

| Parametr `plugins.HeatMap` | Opis |
|---|---|
| `data` ✅ | Lista `[lat, lon]` lub `[lat, lon, waga]` (waga domyślnie 1) |
| `radius` ⬜ | Promień punktu w px (25) |
| `blur` ⬜ | Rozmycie (15); większe = gładsza mapa |
| `min_opacity` ⬜ | Minimalna przezroczystość (0.5) |
| `max_zoom` ⬜ | Zoom, przy którym intensywność osiąga maksimum |
| `gradient` ⬜ | Słownik `{próg_0_1: kolor}`, np. `{0.2: "blue", 0.5: "lime", 0.8: "orange", 1: "red"}` |
| `name`, `overlay`, `control`, `show` ⬜ | Parametry warstwy |

In [ ]:
# ważona (kolumna weight), własny gradient, mniejszy promień i legenda gradientu jako kontrolka (szablon `legend_template` z sekcji 0)
gradient = {0.2: "#2b83ba", 0.45: "#abdda4", 0.65: "#ffffbf", 0.85: "#fdae61", 1.0: "#d7191c"}
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.HeatMap(data=events[["lat", "lon", "weight"]].values.tolist(), radius=14, blur=18, min_opacity=0.35, gradient=gradient,
                name="Wizyty (ważone)").add_to(m)

stops = ", ".join(f"{v} {k * 100:.0f}%" for k, v in gradient.items())
heat_html = (f'<div style="{box_style}"><b>Intensywność wizyt</b>'
             f'<div style="width:160px;height:10px;margin:6px 0 2px;background:linear-gradient(to right, {stops})"></div>'
             f'<div style="display:flex;justify-content:space-between"><span>mała</span><span>duża</span></div></div>')
legend = MacroElement()
legend._template = Template(legend_template.replace("__POS__", "bottomleft").replace("__HTML__", json.dumps(heat_html)))
legend.add_to(m)
folium.LayerControl(collapsed=True).add_to(m)
m

### 9.2. `plugins.HeatMapWithTime`: animacja w czasie

**Kiedy:** zmiana rozkładu przestrzennego w czasie (miesiące, dni). Dane to **lista klatek**, a każda klatka to lista `[lat, lon, waga]`.

#### Wersja podstawowa (wymagane: `data`)

In [ ]:
months = sorted(events["month"].unique())
frames = [events.loc[events["month"] == mo, ["lat", "lon", "weight"]].values.tolist() for mo in months]

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.HeatMapWithTime(data=frames, index=months).add_to(m)
m

| Parametr `plugins.HeatMapWithTime` | Opis |
|---|---|
| `data` ✅ | Lista klatek (każda: lista `[lat, lon, waga]`) |
| `index` ⬜ | Etykiety klatek (np. miesiące) |
| `radius` ⬜ | Promień (15) |
| `min_opacity`, `max_opacity` ⬜ | Zakres przezroczystości |
| `scale_radius` ⬜ | Skaluje promień z zoomem |
| `gradient` ⬜ | Gradient jak w `HeatMap` |
| `use_local_extrema` ⬜ | `True` = normalizacja w każdej klatce (porównuj kształt, nie wartości); `False` = wspólna skala |
| `auto_play` ⬜ | Start animacji od razu |
| `display_index` ⬜ | Pokaż etykietę klatki |
| `speed_slider`, `min_speed`, `max_speed`, `speed_step` ⬜ | Suwak prędkości |
| `position` ⬜ | Pozycja odtwarzacza (`"bottomleft"`) |

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.HeatMapWithTime(data=frames, index=months, radius=14, min_opacity=0.3, max_opacity=0.9, gradient=gradient,
                        use_local_extrema=False, auto_play=False, display_index=True, position="bottomleft", name="Wizyty w czasie").add_to(m)
m

### 9.3. `plugins.TimestampedGeoJson`: zdarzenia punktowe w czasie

Każdy punkt ma czas (`properties.time`) i pojawia się w odpowiednim momencie. Dobre do tras, zdarzeń, wizyt.

| Parametr `plugins.TimestampedGeoJson` | Opis |
|---|---|
| `data` ✅ | GeoJSON `FeatureCollection` z `properties.time` (ISO 8601 lub epoch ms) |
| `period` ⬜ | Krok czasu ISO 8601: `"P1D"` (dzień), `"P7D"`, `"P1M"` |
| `duration` ⬜ | Jak długo punkt jest widoczny (np. `"P14D"`) |
| `add_last_point` ⬜ | Zostaw ostatni punkt trasy |
| `auto_play`, `loop`, `loop_button` ⬜ | Odtwarzanie |
| `max_speed`, `min_speed`, `transition_time` ⬜ | Szybkość |
| `date_options` ⬜ | Format daty na pasku (`"YYYY-MM-DD"`) |
| `time_slider_drag_update` ⬜ | Aktualizacja podczas przeciągania |
| `duration`, `speed_slider` ⬜ | |

In [ ]:
ev = events.sample(400, random_state=1).sort_values("date")
features = [{
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [r.lon, r.lat]},                     # GeoJSON: [lon, lat]!
    "properties": {"time": r.date.strftime("%Y-%m-%d"), "popup": f"Waga: {r.weight}",
                   "icon": "circle", "iconstyle": {"fillColor": "#e76f51", "fillOpacity": 0.7, "stroke": "false", "radius": 3 + r.weight}},
} for r in ev.itertuples()]

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.TimestampedGeoJson({"type": "FeatureCollection", "features": features}, period="P7D", duration="P14D",
                           add_last_point=False, auto_play=False, loop=False, max_speed=3, loop_button=True,
                           date_options="YYYY-MM-DD", time_slider_drag_update=True).add_to(m)
m

### 9.4. `plugins.TimeSliderChoropleth`: choropleth zmienny w czasie

Wymaga słownika stylów `{id_obiektu: {czas_epoch: {"color": hex, "opacity": x}}}`, gdzie `id_obiektu` to `id` cechy w GeoJSON (dla GeoDataFrame: indeks jako tekst), a czas to sekundy epoch jako **tekst**. Poniżej wartości są losowe (pokazują mechanikę).

In [ ]:
month_ts = pd.date_range("2025-01-01", periods=12, freq="MS")
vals = pd.DataFrame(rng.integers(50, 500, (len(voiv), 12)), index=voiv.index, columns=month_ts)       # województwo × miesiąc (dane przykładowe)
cmap_t = cm.linear.YlGnBu_09.scale(vals.to_numpy().min(), vals.to_numpy().max())
cmap_t.caption = "Wartość miesięczna (dane losowe)"

styledict = {str(i): {str(int(ts.timestamp())): {"color": cmap_t(vals.loc[i, ts]), "opacity": 0.85} for ts in month_ts} for i in vals.index}

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
plugins.TimeSliderChoropleth(data=voiv.to_json(), styledict=styledict, name="Choropleth w czasie").add_to(m)
cmap_t.add_to(m)
m

## 10. Wtyczki (`folium.plugins`)

Najczęściej używane dodatki do interfejsu. Każdy ma parametr `position` (róg mapy), więc rozmieszczaj je zgodnie z tabelą z sekcji 7.2.

| Wtyczka | Do czego | Wymagane ✅ / ważne opcjonalne ⬜ |
|---|---|---|
| `Fullscreen` | Tryb pełnoekranowy | ⬜ `position`, `title`, `title_cancel`, `force_separate_button` |
| `MiniMap` | Mała mapa orientacyjna | ⬜ `tile_layer`, `position`, `width`, `height`, `zoom_level_offset`, `toggle_display`, `minimized` |
| `MeasureControl` | Pomiar odległości i powierzchni | ⬜ `position`, `primary_length_unit` (`"meters"`, `"kilometers"`), `primary_area_unit` (`"sqmeters"`, `"hectares"`, `"sqkilometers"`), `secondary_*` |
| `MousePosition` | Współrzędne kursora | ⬜ `position`, `separator`, `num_digits`, `prefix`, `lng_first`, `lat_formatter`, `lng_formatter` |
| `LocateControl` | Lokalizacja użytkownika (GPS) | ⬜ `auto_start`, `position`, `strings` |
| `Geocoder` | Wyszukiwanie adresów (Nominatim, wymaga internetu) | ⬜ `collapsed`, `position`, `add_marker` |
| `Draw` | Rysowanie obiektów (punkty, linie, poligony) z eksportem do GeoJSON | ⬜ `export`, `filename`, `position`, `draw_options`, `edit_options` |
| `Search` | Wyszukiwanie obiektów w warstwie GeoJson po atrybucie | ✅ `layer`; ⬜ `geom_type`, `search_label`, `search_zoom`, `placeholder`, `collapsed`, `position` |
| `TagFilterButton` | Filtrowanie markerów po tagach | ✅ `data` (lista tagów); markery z `tags=[...]` |
| `BeautifyIcon` | Ładniejsze ikony (numery, kształty) | ⬜ `icon`, `icon_shape`, `border_color`, `text_color`, `background_color`, `number`, `inner_icon_style` |
| `SideBySideLayers` | Suwak porównujący dwie warstwy (np. ortofoto i mapa) | ✅ `layer_left`, `layer_right` |
| `DualMap` | Dwie zsynchronizowane mapy obok siebie | ⬜ `location`, `zoom_start`, `layout`, `tiles` |
| `Terminator` | Linia dnia i nocy | brak |

In [ ]:
# panel narzędzi: kontrolki w rogach zgodnie z zasadą (bez nakładania)
m = folium.Map(location=POLAND, zoom_start=6, tiles="OpenStreetMap")
plugins.Fullscreen(position="topleft", title="Pełny ekran", title_cancel="Zamknij pełny ekran").add_to(m)
plugins.MeasureControl(position="topleft", primary_length_unit="kilometers", primary_area_unit="sqkilometers").add_to(m)
plugins.Geocoder(collapsed=True, position="topleft", add_marker=True).add_to(m)
plugins.LocateControl(auto_start=False, position="topleft").add_to(m)
plugins.MousePosition(position="bottomleft", separator=" | ", prefix="Lat | Lon:", num_digits=4).add_to(m)
plugins.MiniMap(tile_layer="OpenStreetMap", position="bottomright", width=130, height=130, toggle_display=True, zoom_level_offset=-5).add_to(m)
plugins.Draw(export=True, filename="rysunek.geojson", position="topleft",
             draw_options={"polyline": True, "polygon": True, "rectangle": True, "circle": False, "marker": True, "circlemarker": False},
             edit_options={"edit": True}).add_to(m)
m

In [ ]:
# Search: wyszukiwanie sklepu po ID (warstwa GeoJson jest przeszukiwana po polu search_label)
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
stores_layer = folium.GeoJson(gdf_stores[["store_id", "city", "geometry"]], name="Sklepy",
                              marker=folium.CircleMarker(radius=4, color="white", weight=0.8, fill=True, fill_color="#264653", fill_opacity=0.9),
                              tooltip=folium.GeoJsonTooltip(fields=["store_id", "city"])).add_to(m)
plugins.Search(layer=stores_layer, geom_type="Point", placeholder="Szukaj sklepu (np. S0042)", collapsed=False,
               search_label="store_id", search_zoom=13, position="topleft").add_to(m)
m

In [ ]:
# TagFilterButton: filtr markerów wg formatu; BeautifyIcon: numer rankingu
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
for r in gdf_stores.head(150).itertuples():
    folium.Marker([r.lat, r.lon], tags=[r.format], tooltip=f"{r.store_id} ({r.format})",
                  icon=folium.Icon(color=color_by_format[r.format], icon="shopping-cart", prefix="fa")).add_to(m)
plugins.TagFilterButton(["Mały", "Średni", "Duży"], clear_text="Pokaż wszystkie", position="topleft").add_to(m)

for rank, r in enumerate(gdf_stores.nlargest(5, "revenue").itertuples(), start=1):          # top 5 z numerami
    folium.Marker([r.lat, r.lon], tooltip=f"#{rank}: {r.store_id}, {r.revenue:,.0f} tys. PLN",
                  icon=plugins.BeautifyIcon(number=rank, icon_shape="circle", border_color="#e76f51", text_color="#e76f51",
                                            background_color="white", inner_icon_style="font-size:13px;")).add_to(m)
m

In [ ]:
# SideBySideLayers: suwak porównujący dwa podkłady
m = folium.Map(location=[52.23, 21.01], zoom_start=12, tiles=None)
left = folium.TileLayer("OpenStreetMap", name="Mapa").add_to(m)
right = folium.TileLayer("Esri.WorldImagery", name="Satelita").add_to(m)
plugins.SideBySideLayers(layer_left=left, layer_right=right).add_to(m)
m

# DualMap: dwie zsynchronizowane mapy (te same dane, dwie miary)
dual = plugins.DualMap(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas", layout="horizontal")
for sub_map, col, cmap_x in [(dual.m1, "revenue_sum", cm.linear.YlOrRd_09), (dual.m2, "n_stores", cm.linear.Blues_09)]:
    scale = cmap_x.scale(voiv_stats[col].min(), voiv_stats[col].max())
    folium.GeoJson(voiv_stats, style_function=lambda f, c=col, s=scale: {"fillColor": s(f["properties"][c]) if f["properties"][c] is not None else "#e0e0e0",
                                                                         "color": "white", "weight": 1, "fillOpacity": 0.85},
                   tooltip=folium.GeoJsonTooltip(fields=["nazwa", col])).add_to(sub_map)
dual

## 11. Tytuł, CSS/JS, raster i układ wielu map

### 11.1. Tytuł mapy i własny CSS

Tytuł umieszczamy **na środku u góry** (rogi zajmują kontrolki). `pointer-events: none` sprawia, że nie blokuje klikania mapy pod spodem.

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
title_html = ('<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;pointer-events:none;'
              'background:rgba(255,255,255,.92);padding:5px 14px;border-radius:6px;box-shadow:0 1px 6px rgba(0,0,0,.3);'
              'font:600 15px Arial;color:#264653;white-space:nowrap">Sieć sklepów: przychód wg województw</div>')
m.get_root().html.add_child(folium.Element(title_html))

css = "<style>.leaflet-tooltip{font:12px Arial;border-radius:4px} .leaflet-container{background:#eef3f6}</style>"      # własny CSS globalnie
m.get_root().header.add_child(folium.Element(css))
m

### 11.2. Opcje Leaflet i proste zdarzenia

`folium.Map` przyjmuje dodatkowe opcje Leaflet jako `**kwargs`, a małe obiekty pomocnicze dodają interakcje:

| Element | Efekt |
|---|---|
| `folium.Map(..., scrollWheelZoom=False)` | Wyłącza zoom kółkiem (przydatne, gdy mapa jest osadzona w długiej stronie) |
| `folium.Map(..., dragging=False)` | Blokada przesuwania |
| `folium.LatLngPopup()` | Klik pokazuje współrzędne punktu |
| `folium.ClickForMarker(popup="Punkt")` | Klik dodaje marker |
| `plugins.ScrollZoomToggler()` | Przycisk włączający zoom kółkiem |

In [ ]:
m = folium.Map(location=POLAND, zoom_start=6, scrollWheelZoom=False)
m.add_child(folium.LatLngPopup())                       # klik: lat/lon
m.add_child(folium.ClickForMarker(popup="Nowy punkt"))
m

# własny JavaScript przez MacroElement: np. reakcja na zmianę przybliżenia (podgląd w konsoli przeglądarki)
js = MacroElement()
js._template = Template("""
{% macro script(this, kwargs) %}
{{ this._parent.get_name() }}.on('zoomend', function () { console.log('zoom =', {{ this._parent.get_name() }}.getZoom()); });
{% endmacro %}
""")
js.add_to(m)

### 11.3. Raster: `ImageOverlay` (np. gęstość z NumPy)

Tablicę (np. wynik `np.histogram2d`) nakładamy jako obraz na zadany prostokąt geograficzny.

| Parametr `folium.raster_layers.ImageOverlay` | Opis |
|---|---|
| `image` ✅ | URL obrazu lub tablica NumPy (2D z `colormap`, 3D RGB/RGBA) |
| `bounds` ✅ | `[[lat_min, lon_min], [lat_max, lon_max]]` |
| `opacity` ⬜ | Przezroczystość |
| `origin` ⬜ | `"upper"` (wiersz 0 = północ) lub `"lower"` |
| `colormap` ⬜ | Funkcja wartość → RGBA (dla tablic 2D) |
| `mercator_project` ⬜ | Korekta rzutu Mercatora (zalecane dla dużych obszarów) |
| `name`, `overlay`, `control`, `show` ⬜ | |

In [ ]:
# siatka gęstości sklepów (histogram 2D) jako raster z przezroczystością w pustych komórkach
lat_edges = np.linspace(49.0, 55.0, 120)
lon_edges = np.linspace(14.0, 24.2, 170)
H, _, _ = np.histogram2d(gdf_stores["lat"], gdf_stores["lon"], bins=[lat_edges, lon_edges])         # H[lat_bin, lon_bin]

rgba = plt.get_cmap("YlOrRd")(H / H.max())                                                         # kolory z colormapy matplotlib
rgba[..., 3] = np.where(H > 0, 0.75, 0)                                                            # puste komórki przezroczyste

m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.raster_layers.ImageOverlay(image=rgba, bounds=[[49.0, 14.0], [55.0, 24.2]], origin="lower", mercator_project=True, opacity=1, name="Gęstość sklepów").add_to(m)
folium.LayerControl(collapsed=True).add_to(m)
m

> Raster z NumPy jest obrazem: nie ma tooltipów i nie skaluje się przy zoomie (rozdzielczość = rozmiar tablicy). Dla analiz rastrowych (GeoTIFF, NDVI) lepsze są `rasterio` + eksport do PNG z geokodowaniem lub kafelki (`gdal2tiles`) i `TileLayer`.

### 11.4. Wiele map na jednej stronie i osadzanie

In [ ]:
# dwie niezależne mapy obok siebie w jednym obiekcie Figure
fig = folium.Figure(width=1000, height=420)
m1 = folium.Map(location=POLAND, zoom_start=6, tiles="OpenStreetMap")
m2 = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldImagery")
m1.add_to(fig.add_subplot(1, 2, 1))                      # add_subplot(wiersze, kolumny, indeks)
m2.add_to(fig.add_subplot(1, 2, 2))
fig

In [ ]:
# osadzanie zapisanej mapy w notebooku (np. przy bardzo dużych mapach lub kilku mapach naraz)
from IPython.display import IFrame
m = folium.Map(location=POLAND, zoom_start=6)
m.save("mapa_iframe.html")
IFrame("mapa_iframe.html", width="100%", height=450)

W aplikacjach: **Streamlit** (`streamlit-folium`: `st_folium(m, height=500, returned_objects=["last_object_clicked"])` zwraca kliknięcia do Pythona), **Dash** (`dash-leaflet` jako natywny odpowiednik), **Flask/FastAPI** (`m.get_root().render()` jako odpowiedź HTML).

## 12. Integracje: GeoPandas/CRS, QGIS, SQL Server, BigQuery, Airflow

### 12.1. Układy współrzędnych (CRS) w praktyce

| Zadanie | Układ | Uwagi |
|---|---|---|
| Wyświetlanie w Folium | EPSG:4326 | `gdf.to_crs(4326)` przed `GeoJson`; surowe współrzędne (`Marker`) podawaj jako `[lat, lon]` |
| Odległości, bufory, pola (Polska) | EPSG:2180 (PL-1992, metry) | Dla lokalnych analiz także PL-2000: 2176–2179; globalnie UTM (np. 32633/32634) |
| Dane z geoportali i GUS | często EPSG:2180 lub 2178/2177 | Sprawdź `gdf.crs`; brak CRS → `gdf.set_crs(...)` (ustawia etykietę, **nie** przelicza) |
| Przeliczenie | `gdf.to_crs(nowy)` | Faktycznie transformuje współrzędne |

In [ ]:
# dane z układu metrycznego (np. z QGIS/geoportalu): x, y w EPSG:2180 → mapa WGS84
pts_2180 = gpd.GeoDataFrame({"name": ["A", "B"]}, geometry=gpd.points_from_xy([638_000, 500_000], [486_000, 250_000]), crs="EPSG:2180")
pts_4326 = pts_2180.to_crs(4326)
print(pts_4326.geometry.apply(lambda g: (round(g.y, 4), round(g.x, 4))).tolist())        # (lat, lon): Warszawa/Kraków okolice

m = folium.Map(location=POLAND, zoom_start=6)
folium.GeoJson(pts_4326, marker=folium.CircleMarker(radius=8), tooltip=folium.GeoJsonTooltip(fields=["name"])).add_to(m)
m

### 12.2. Agregacja do siatki (zamiast tysięcy punktów)

Dla dużych zbiorów punktów zamień punkty na **komórki siatki** z licznikiem/sumą i pokaż jako choropleth. Mapa jest lekka, a wynik czytelniejszy. (Alternatywa: siatka heksagonalna H3, pakiet `h3`.)

In [ ]:
from shapely.geometry import box

cell = 25_000                                                          # bok komórki: 25 km (EPSG:2180, metry)
pts = gdf_stores.to_crs(2180)
minx, miny, maxx, maxy = pts.total_bounds
xs = np.arange(minx - cell, maxx + cell, cell)
ys = np.arange(miny - cell, maxy + cell, cell)
grid = gpd.GeoDataFrame(geometry=[box(x, y, x + cell, y + cell) for x in xs for y in ys], crs=2180)
grid["cell_id"] = grid.index

hit = gpd.sjoin(pts[["revenue", "geometry"]], grid, predicate="within")
cells = hit.groupby("cell_id").agg(n=("revenue", "size"), revenue=("revenue", "sum")).reset_index()
grid_stats = grid.merge(cells, on="cell_id").to_crs(4326)               # tylko komórki z danymi, z powrotem do WGS84

cmap_g = cm.linear.YlGnBu_09.scale(grid_stats["revenue"].min(), grid_stats["revenue"].max())
cmap_g.caption = "Przychód w komórce 25×25 km [tys. PLN]"
m = folium.Map(location=POLAND, zoom_start=6, tiles="Esri.WorldGrayCanvas")
folium.GeoJson(grid_stats[["n", "revenue", "geometry"]], name="Siatka 25 km",
               style_function=lambda f: {"fillColor": cmap_g(f["properties"]["revenue"]), "color": "white", "weight": 0.5, "fillOpacity": 0.8},
               highlight_function=lambda f: {"weight": 2, "color": "#264653"},
               tooltip=folium.GeoJsonTooltip(fields=["n", "revenue"], aliases=["Sklepy", "Przychód"], localize=True)).add_to(m)
cmap_g.add_to(m)
m

### 12.3. QGIS ↔ Python ↔ Folium

| Cel | Rozwiązanie |
|---|---|
| Wymiana danych | **GeoPackage** (`.gpkg`: wiele warstw, CRS, atrybuty, bez limitu nazw kolumn jak w Shapefile). `gdf.to_file("dane.gpkg", layer="sklepy", driver="GPKG")` → w QGIS: *Warstwa → Dodaj warstwę → wektorowa* |
| Wczytanie wyniku analizy z QGIS | `gpd.read_file("dane.gpkg", layer="strefy")` |
| Interaktywna mapa www bezpośrednio z QGIS | Wtyczka **qgis2web** (eksport do Leaflet/OpenLayers z zachowaniem symbolizacji) |
| Zaawansowana kartografia, layouty do druku | QGIS (kompozytor wydruku); Folium jest do map interaktywnych |
| Powtarzalna automatyzacja analiz | Python/GeoPandas (skrypty, Airflow), a w QGIS tylko wizualne sprawdzenie wyników |
| Procesing QGIS w skrypcie | `qgis_process run native:buffer ...` lub PyQGIS (środowisko QGIS) |

In [ ]:
import tempfile, os
path = os.path.join(tempfile.gettempdir(), "dane_sklepy.gpkg")
gdf_stores[["store_id", "city", "format", "revenue", "geometry"]].to_file(path, layer="sklepy", driver="GPKG")       # zapis (do otwarcia w QGIS)
voiv_stats.to_file(path, layer="wojewodztwa", driver="GPKG")                                                          # druga warstwa w tym samym pliku
print(gpd.list_layers(path))                                                                                          # lista warstw w pliku

back = gpd.read_file(path, layer="sklepy")                                                                            # odczyt
print(back.crs, len(back))

### 12.4. SQL Server i BigQuery: geometrie z baz danych

Dwa typowe układy danych: (1) **kolumny `lat`/`lon`** (najprościej: `gpd.points_from_xy`) i (2) **kolumny przestrzenne** (`geography`/`geometry` w SQL Server, `GEOGRAPHY` w BigQuery). Kolumny przestrzenne odczytujesz jako **WKT** i zamieniasz na geometrie.

| Silnik | Zapytanie zwracające WKT | Uwagi |
|---|---|---|
| SQL Server | `SELECT store_id, geom.STAsText() AS wkt, geom.STSrid AS srid FROM dbo.stores` | Dla typu `geography` WKT ma kolejność **`POINT(lon lat)`**; sprawdź SRID (zwykle 4326) |
| BigQuery | `SELECT store_id, ST_ASTEXT(geog) AS wkt FROM dataset.stores` | Nowsze klienty mogą zwracać geometrię wprost jako Shapely (`to_dataframe(geography_as_object=True)`; sprawdź wersję biblioteki) |

In [ ]:
# wynik zapytania SQL (tu: przykładowa ramka z kolumną WKT, tak jak zwróciłby ją cursor/pandas.read_sql)
sql = "SELECT store_id, geom.STAsText() AS wkt FROM dbo.stores"            # SQL Server (wykonanie np. przez mssql_python → DataFrame)
db_df = pd.DataFrame({"store_id": ["S1", "S2", "S3"],
                      "wkt": ["POINT (21.0122 52.2297)", "POINT (19.9450 50.0647)", "POINT (17.0385 51.1079)"]})

db_gdf = gpd.GeoDataFrame(db_df, geometry=gpd.GeoSeries.from_wkt(db_df["wkt"]), crs="EPSG:4326").drop(columns="wkt")
m = folium.Map(location=POLAND, zoom_start=6)
folium.GeoJson(db_gdf, marker=folium.CircleMarker(radius=7, color="#e76f51", fill=True), tooltip=folium.GeoJsonTooltip(fields=["store_id"])).add_to(m)
m

> Agregację (liczba obiektów w siatce/województwie, sumy) warto wykonać **w SQL** (`STIntersects`, `GROUP BY`; w BigQuery `ST_INTERSECTS`, `ST_GEOGPOINT`), a do Pythona pobrać wynik. Mapa dostaje wtedy setki wierszy zamiast milionów.

### 12.5. Automatyczny raport mapowy w Airflow

Folium dobrze nadaje się do **raportu HTML generowanego cyklicznie**: zadanie DAG-a pobiera dane (SQL), buduje mapę i zapisuje `mapa.html` (np. na dysk sieciowy, do bucketa, jako załącznik maila). Poniższy przykład jest szkicem (wymaga środowiska Airflow; nie uruchamiaj go w tym notebooku).

In [ ]:
# --- szkic DAG-a (plik dags/mapa_sklepow.py); Airflow 3: from airflow.sdk import dag, task | Airflow 2: from airflow.decorators import dag, task
from datetime import datetime
from airflow.sdk import dag, task

@dag(schedule="0 6 * * *", start_date=datetime(2026, 1, 1), catchup=False, tags=["mapy"])
def mapa_sklepow():
    @task
    def pobierz_dane() -> str:
        # SELECT ... FROM dbo.stores (SQL Server) → DataFrame → zapis parquet; zwracamy tylko ścieżkę (XCom nie powinien nieść dużych danych)
        path = "/data/stores.parquet"
        return path

    @task
    def zbuduj_mape(path: str) -> str:
        import pandas as pd, folium
        df = pd.read_parquet(path)
        m = folium.Map(location=[52.0, 19.4], zoom_start=6)
        for r in df.itertuples():
            folium.CircleMarker([r.lat, r.lon], radius=4).add_to(m)
        out = "/reports/mapa.html"
        m.save(out)
        return out

    zbuduj_mape(pobierz_dane())

mapa_sklepow()

## 13. Zapis, wydajność, pułapki

### 13.1. Wydajność i rozmiar pliku

Plik HTML zawiera **wszystkie dane** (współrzędne, atrybuty, popupy). Rozmiar rośnie z liczbą wierzchołków i tekstu.

| Problem | Rozwiązanie |
|---|---|
| Ciężkie poligony (granice) | `gdf.geometry.simplify(tolerance, preserve_topology=True)`; tolerancja w stopniach (≈ 0.005° ≈ 500 m) lub w metrach po `to_crs(2180)` |
| Zbyt dokładne współrzędne | `shapely.set_precision(gdf.geometry, 0.0001)` (≈ 10 m) zmniejsza plik |
| Wiele kolumn w GeoJson | Przekaż tylko potrzebne: `gdf[["kol1", "kol2", "geometry"]]` |
| Tysiące markerów | `CircleMarker` + `prefer_canvas=True`, `MarkerCluster`/`FastMarkerCluster`, heatmapa lub agregacja do siatki |
| Wiele popupów z HTML/obrazami | `folium.Popup(..., lazy=True)`, wykresy tylko dla kluczowych obiektów |
| Wiele warstw na starcie | `show=False` dla ciężkich warstw (nadal są w pliku, ale nie rysują się) |
| Bardzo duże mapy (> 20–30 MB) | Zapis do pliku i `IFrame`; agregacja po stronie SQL; WebGL: `pydeck`/`lonboard` |

In [ ]:
# porównanie rozmiaru: województwa pełne vs uproszczone
full = gpd.read_file(io.BytesIO(urllib.request.urlopen(url, timeout=30).read()))
for label, g in [("pełne", full), ("uproszczone 0.005°", full.assign(geometry=full.geometry.simplify(0.005, preserve_topology=True)))]:
    mm = folium.Map(location=POLAND, zoom_start=6)
    folium.GeoJson(g[["nazwa", "geometry"]]).add_to(mm)
    print(f"{label:>20}: {len(mm.get_root().render()) / 1024:,.0f} KB")

### 13.2. Zapis i eksport

| Cel | Sposób |
|---|---|
| Interaktywny HTML | `m.save("mapa.html")` (kafelki ładują się z internetu) |
| Fragment HTML do raportu | `m.get_root().render()` lub `m._repr_html_()` |
| Statyczny obraz (PNG) | `m._to_png(delay=5)` (wymaga `selenium` i przeglądarki/sterownika) lub zrzut ekranu; do raportów PDF lepsze `matplotlib` + `contextily` (podkład) / `gdf.plot()` |
| Dane z narysowanych obiektów | `plugins.Draw(export=True)` (przycisk eksportu GeoJSON), a w aplikacji `streamlit-folium` zwraca `all_drawings` |

### 13.3. Najczęstsze pułapki

1. **Kolejność współrzędnych:** Folium `[lat, lon]`, GeoJSON/Shapely `(lon, lat)`. Markery „w złym miejscu" prawie zawsze znaczą zamienione osie.
2. **Zły CRS:** współrzędne w EPSG:2180 (setki tysięcy) jako `lat/lon` dadzą pustą mapę. GeoDataFrame przekazany do `GeoJson` jest konwertowany na 4326, ale surowe listy `[x, y]` już nie.
3. **Błędy serializacji JSON:** `Timestamp`, `NaT`, `NaN`, typy NumPy (`np.int64` w niektórych kontekstach) mogą wywołać `TypeError`. Zamień daty na tekst (`strftime`), a `NaN` obsłuż (`fillna` lub `None`).
4. **Choropleth: klucze się nie zgadzają** (polskie znaki, spacje, wielkość liter, `"Mazowieckie"` vs `"mazowieckie"`): wszystko dostaje kolor braków. Sprawdź `set(data["klucz"]) - set(gdf["klucz"])` oraz poprawność `key_on`.
5. **Progi (`bins`) nie obejmują danych:** ustaw je tak, by pokrywały całą rozpiętość wartości.
6. **Kafelki CartoDB bez klucza** (zob. sekcja 1): używaj `OpenStreetMap` lub `Esri.*`; sprawdzaj warunki użycia podkładu przy publikacji.
7. **Legendy `position: fixed`** nachodzą na kontrolki: używaj kontrolek Leaflet (sekcja 7).
8. **`LayerControl` dodany przed warstwami** nie pokaże późniejszych warstw: dodawaj go na końcu.
9. **Kolejność rysowania = kolejność dodawania:** poligony najpierw, punkty na wierzchu.
10. **Mapa się nie wyświetla** w notebooku: obiekt mapy musi być ostatnim wyrażeniem komórki (lub `display(m)`); w VS Code zaufaj notebookowi (Trust). Bardzo duże mapy zapisz do pliku.
11. **Popup vs Tooltip:** tooltip = najechanie, popup = kliknięcie. Na urządzeniach dotykowych tooltip działa jak popup.
12. **Zoom/wydajność na telefonach:** unikaj tysięcy `Marker` (DOM), stosuj canvas lub klastry.

## 14. Ściągawka i porównanie z innymi bibliotekami

### 14.1. Które narzędzie do czego (Folium)

| Chcę pokazać… | Użyj |
|---|---|
| Kilka ważnych punktów z opisem | `Marker` + `Popup`/`Tooltip` |
| Wartość punktu jako rozmiar/kolor | `CircleMarker` (`prefer_canvas=True`) + legenda kontrolką |
| Setki–tysiące punktów | `MarkerCluster`; od ~5 000 `FastMarkerCluster` lub heatmapa |
| Gęstość / „gorące punkty" | `HeatMap` (waga w 3. kolumnie) |
| Zmianę w czasie | `HeatMapWithTime`, `TimestampedGeoJson`, `TimeSliderChoropleth` |
| Wskaźnik na obszarach | `Choropleth` lub `GeoJson` + `branca` (+ `mapclassify`) |
| Trasy, przepływy | `PolyLine`, `AntPath` |
| Strefy zasięgu | bufory w EPSG:2180 → `GeoJson`; `Circle` dla prostych kół |
| Porównanie dwóch map | `DualMap`, `SideBySideLayers` |
| Szybki podgląd GeoDataFrame | `gdf.explore(column=..., scheme=...)` |
| Filtrowanie/wyszukiwanie | `TagFilterButton`, `Search`, `LayerControl`/`GroupedLayerControl` |

### 14.2. Folium czy inna biblioteka?

| Biblioteka | Mocna strona | Kiedy zamiast Folium |
|---|---|---|
| **Folium** | Leaflet jako statyczny HTML, prosty, ogromna liczba wtyczek | Raporty HTML, notebooki, szybkie mapy tematyczne i punktowe (do kilkudziesięciu tysięcy obiektów) |
| **GeoPandas `.explore()`** | Skrót do Folium jednym wywołaniem | Eksploracja GeoDataFrame (to ten sam silnik) |
| **ipyleaflet** | Leaflet jako widget Jupyter: dwukierunkowa komunikacja (zdarzenia kliknięć w Pythonie, aktualizacje na żywo) | Interaktywna analiza w notebooku z reakcją Pythona na akcje użytkownika |
| **pydeck / lonboard / kepler.gl** | WebGL (deck.gl): setki tysięcy–miliony punktów, 3D, łuki, heksagony | Duże zbiory, wizualizacje 3D i gęstości; `lonboard` współpracuje z GeoPandas przez GeoArrow |
| **Plotly (`px.scatter_map`, `px.choropleth_map`)** | Spójność z wykresami i dashboardami Plotly/Dash | Gdy mapa jest częścią dashboardu z innymi wykresami Plotly (zob. przewodnik Plotly) |
| **leafmap / geemap** | Nakładki na Folium/ipyleaflet, Google Earth Engine, dane rastrowe | Analizy satelitarne i rastrowe |
| **Matplotlib + GeoPandas + contextily** | Statyczne mapy do PDF/publikacji | Raporty drukowane, pełna kontrola typografii |
| **QGIS** | Zaawansowana symbolizacja, kompozytor wydruku, analizy GIS | Mapy kartograficzne i interaktywna praca z warstwami; `qgis2web` eksportuje Leaflet |
| **Power BI** | Mapy w raportach BI (Azure Maps / ArcGIS visual) | Folium nie działa w wizualizacjach Python Power BI (zwracają statyczny obraz) |

### Co dalej (rozwój w kierunku Twojego stacku)

- **Streamlit + `streamlit-folium`:** dashboard mapowy nad SQL Server/BigQuery, w którym kliknięcie na mapie zwraca obiekt do Pythona i filtruje tabelę.
- **DuckDB spatial:** agregacje przestrzenne (`ST_Within`, `ST_Distance`) w silniku kolumnowym i tylko wynik na mapę.
- **H3 (`h3-py`):** heksagonalna agregacja punktów zamiast prostokątnej siatki; wyniki jako `GeoJson` w Folium.
- **Airflow:** cykliczny raport HTML z mapą (zob. 12.5) wysyłany mailem lub publikowany.
- **OSMnx / routing:** sieci drogowe, czasy dojazdu (izochrony) jako poligony w Folium.
- **Python + QGIS:** powtarzalne przetwarzanie w GeoPandas, wizualna kontrola i kartografia w QGIS, interaktywny wynik w Folium.